# What’s 4 Dinner? — Open-Source Research Pipeline (v1.2.2)
Uses DuckDuckGo + Qwen2.5-7B on Colab T4 GPU. Zero API Keys Required.

In [ ]:
# Step 1: Install packages (Torch is left intact to avoid CUDA errors)
!nvidia-smi
!pip install -q -U transformers accelerate bitsandbytes duckduckgo-search

In [ ]:
# Step 2: Web Search for Summerville, SC
from duckduckgo_search import DDGS
import json

LOCATION = "Summerville, SC"
SEARCH_QUERY = f"healthy dinner restaurant menu {LOCATION} grilled lean protein seafood"

print(f"Searching menus in {LOCATION}...")
search_context = ""
with DDGS() as ddgs:
    results = list(ddgs.text(SEARCH_QUERY, max_results=6))
    for i, r in enumerate(results, 1):
        search_context += f"Source [{i}]: {r.get('title','')} - {r.get('body','')}\n"
print("Search context retrieved!")

In [ ]:
# Step 3: Load Model & Suppress Deprecation Warnings
import warnings
import os
import torch
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

model_id = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)

print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map='auto')
generator = pipeline('text-generation', model=model, tokenizer=tokenizer)
print("Model ready!")

In [ ]:
# Step 4: Generate meals.json (Fast, Non-Looping)
import re
sys_prompt = """You are a clinical dietetic menu auditor. 
Recommend healthy Lowcountry dinner options that meet:
1. GLP-1 therapy suitable (lean protein, low fat, non-nausea).
2. Low Sodium (< 500mg).
3. Low Calorie (< 450 kcal).
4. High Protein (> 30g).
Output ONLY a JSON array with objects matching: [{"id":"1","type":"dine-out","venue":"Name","location":"Summerville, SC","title":"Entree","calories":380,"protein":36,"sodium":340,"glp1":true,"lowSodium":true,"lowCal":true,"highProtein":true,"orderTip":"Tip","address":"Full Street Address or City"}]. No markdown commentary."""

user_prompt = f"Target Location: {LOCATION}\nWeb snippets:\n{search_context}\nGenerate 6 dinner options."
messages = [{'role':'system','content':sys_prompt}, {'role':'user','content':user_prompt}]
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("Generating meals.json (takes ~20-30 secs)...")
out = generator(prompt_text, max_new_tokens=900, do_sample=True, temperature=0.3, pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)
raw_res = out[0]['generated_text'][len(prompt_text):].strip()

m = re.search(r'\[\s*\{.*\}\s*\]', raw_res, re.DOTALL)
clean = m.group(0) if m else raw_res.replace('```json','').replace('```','').strip()

parsed = json.loads(clean)
with open('meals.json', 'w') as f:
    json.dump(parsed, f, indent=2)
print("Successfully saved meals.json!")

In [ ]:
# Step 5: Download meals.json from Colab
from google.colab import files
files.download('meals.json')